Got it ✅
You want me to **restructure the project scaffolding** to reflect a **clean separation of configuration, MCP servers, tools, LLM, and agents**, where everything is driven dynamically from config.

Here’s the proposed **hierarchy + code scaffold**:

---

# 📂 Project Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py         # all agent/mcp/tool/llm configs
│
├── mcp_servers/            # all MCP servers live here
│   ├── __init__.py
│   ├── weather_server.py
│   ├── pollution_server.py
│
├── mcp_clients/            # corresponding MCP clients
│   ├── __init__.py
│   ├── weather_client.py
│   ├── pollution_client.py
│
├── tools/                  # wrappers around MCP clients
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
├── llm/                    # LLM provider wrappers
│   ├── __init__.py
│   ├── llm_provider.py
│
├── agents/                 # factory + parent
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── parent_agent.py
```

---

# ⚙️ `config/settings.py`

```python
# config/settings.py

AGENT_CONFIG = {
    "weather": {
        "mcp_servers": ["weather-mcp-1", "weather-mcp-2"],
        "tools": ["get_city_weather", "get_country_weather"],
        "llm": {"provider": "openai", "model": "gpt-4o-mini"}
    },
    "pollution": {
        "mcp_servers": ["pollution-mcp-1", "pollution-mcp-2"],
        "tools": ["get_city_pollution", "get_country_pollution"],
        "llm": {"provider": "gemini", "model": "gemini-pro"}
    },
    "parent": {
        "llm": {"provider": "openai", "model": "gpt-4o"},
        "agents": ["weather", "pollution"],
        "protocol": "A2A",
        "features": ["RAG", "React", "ChainOfThought"]
    }
}
```

---

# 🌦️ `mcp_servers/weather_server.py`

```python
# mcp_servers/weather_server.py
from fastmcp import FastMCP, Response
import requests

mcp = FastMCP("weather-mcp-1")

@mcp.tool()
def get_weather(location: str) -> Response:
    """Fetch weather info from wttr.in"""
    try:
        resp = requests.get(f"https://wttr.in/{location}?format=3", timeout=5)
        return Response(content=resp.text)
    except Exception as e:
        return Response(content=f"Error: {e}")

if __name__ == "__main__":
    mcp.run()
```

👉 You can duplicate this for `weather-mcp-2` (maybe mock data).

---

# 🏭 `mcp_servers/pollution_server.py`

```python
# mcp_servers/pollution_server.py
from fastmcp import FastMCP, Response

POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

mcp = FastMCP("pollution-mcp-1")

@mcp.tool()
def get_pollution(location: str) -> Response:
    return Response(content=POLLUTION_DATA.get(location, "No data available"))

if __name__ == "__main__":
    mcp.run()
```

👉 Duplicate for `pollution-mcp-2`.

---

# 📡 `mcp_clients/weather_client.py`

```python
# mcp_clients/weather_client.py
from fastmcp import MCPClient

async def fetch_weather(location: str) -> str:
    async with MCPClient("weather-mcp-1") as client:
        result = await client.call("get_weather", location=location)
        return result.content
```

---

# 📡 `mcp_clients/pollution_client.py`

```python
# mcp_clients/pollution_client.py
from fastmcp import MCPClient

async def fetch_pollution(location: str) -> str:
    async with MCPClient("pollution-mcp-1") as client:
        result = await client.call("get_pollution", location=location)
        return result.content
```

---

# 🔧 `tools/weather_tools.py`

```python
# tools/weather_tools.py
from mcp_clients.weather_client import fetch_weather

async def get_city_weather(city: str) -> str:
    return await fetch_weather(city)

async def get_country_weather(country: str) -> str:
    return await fetch_weather(country)
```

---

# 🔧 `tools/pollution_tools.py`

```python
# tools/pollution_tools.py
from mcp_clients.pollution_client import fetch_pollution

async def get_city_pollution(city: str) -> str:
    return await fetch_pollution(city)

async def get_country_pollution(country: str) -> str:
    return await fetch_pollution(country)
```

---

# 🧠 `llm/llm_provider.py`

```python
# llm/llm_provider.py
from openai import OpenAI

def get_llm_client(provider: str, model: str):
    if provider == "openai":
        return OpenAI(), model
    # 🔧 Add support for Gemini/Anthropic here
    raise ValueError(f"Unsupported LLM provider: {provider}")
```

---

# 🏭 `agents/agent_factory.py`

```python
# agents/agent_factory.py
from config.settings import AGENT_CONFIG
from tools import weather_tools, pollution_tools
import asyncio

class AgentFactory:
    def __init__(self):
        self.agents = {}

    def build_agent(self, agent_name: str):
        if agent_name not in AGENT_CONFIG:
            raise ValueError(f"Unknown agent: {agent_name}")

        config = AGENT_CONFIG[agent_name]
        tool_funcs = []

        for tool in config.get("tools", []):
            if hasattr(weather_tools, tool):
                tool_funcs.append(getattr(weather_tools, tool))
            elif hasattr(pollution_tools, tool):
                tool_funcs.append(getattr(pollution_tools, tool))

        async def run(task: dict):
            city = task.get("city")
            country = task.get("country")
            for tool in tool_funcs:
                if city and "city" in tool.__name__:
                    return await tool(city)
                if country and "country" in tool.__name__:
                    return await tool(country)
            return f"{agent_name} agent: No location provided"

        self.agents[agent_name] = run
        return run

    def get_agent(self, agent_name: str):
        return self.agents.get(agent_name) or self.build_agent(agent_name)
```

---

# 🧭 `agents/parent_agent.py` (LLM Router)

```python
# agents/parent_agent.py
from config.settings import AGENT_CONFIG
from llm.llm_provider import get_llm_client

def llm_router(prompt: str) -> str:
    """Route query using configured LLM"""
    llm_conf = AGENT_CONFIG["parent"]["llm"]
    client, model = get_llm_client(llm_conf["provider"], llm_conf["model"])

    routing_prompt = f"""
    You are a router. Decide whether this query is about weather, pollution, or none.

    Query: "{prompt}"

    Respond with only one word: "weather", "pollution", or "end".
    """

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": routing_prompt}],
        max_tokens=1,
    )

    decision = resp.choices[0].message.content.strip().lower()
    return decision if decision in ["weather", "pollution", "end"] else "end"
```

---

# 🚀 `main.py`

```python
# main.py
import asyncio
from langgraph.graph import StateGraph, START, END
from agents.agent_factory import AgentFactory
from agents.parent_agent import llm_router

def build_graph():
    factory = AgentFactory()
    weather_agent = factory.get_agent("weather")
    pollution_agent = factory.get_agent("pollution")

    builder = StateGraph(dict)

    builder.add_node("parent_router", lambda state: {"next": llm_router(state["prompt"])})
    builder.add_node("weather", lambda state: {"result": asyncio.run(weather_agent(state))})
    builder.add_node("pollution", lambda state: {"result": asyncio.run(pollution_agent(state))})

    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda out: out["next"],
        {"weather": "weather", "pollution": "pollution", "end": END}
    )
    builder.add_edge("weather", END)
    builder.add_edge("pollution", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        {"prompt": "What's the weather in Paris?", "city": "Paris"},
        {"prompt": "Check AQI in Delhi", "city": "Delhi"},
        {"prompt": "Tell me a joke"}
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p['prompt']}")
        result = graph.invoke(p)
        print("🤖 Response:", result.get("result"))

if __name__ == "__main__":
    main()
```

---

✅ Now you have:

* **MCP servers** separated (`mcp_servers/`)
* **MCP clients** (`mcp_clients/`)
* **Tools** built on top of clients
* **LLM provider abstraction** (`llm/`)
* **Config-driven agents** (weather, pollution, parent with LLM routing)

---

Do you want me to also make **parent agent itself an MCP server** (so other systems can query it like `parent-mcp` and it will internally route)?
